Project: /meridian/_project.yaml
Book: /meridian/mmm/_book.yaml

<style>
devsite-code .tfo-notebook-code-cell-output {
  max-height: 300px;
  overflow: auto;
  background: rgba(255, 247, 237, 1);  /* orange bg to distinguish from input code cells */
}

devsite-code .tfo-notebook-code-cell-output + .devsite-code-buttons-container button {
  background: rgba(255, 247, 237, .7);  /* orange bg to distinguish from input code cells */
}

devsite-code[dark-code] .tfo-notebook-code-cell-output {
  background: rgba(64, 78, 103, 1);  /* medium slate */
}

devsite-code[dark-code] .tfo-notebook-code-cell-output + .devsite-code-buttons-container button {
  background: rgba(64, 78, 103, .7);  /* medium slate */
}

/* override default table styles for notebook buttons */
.devsite-table-wrapper .tfo-notebook-buttons {
  display: inline-block;
  margin-left: 3px;
  width: auto;
}

.tfo-notebook-buttons tr {
  background: 0;
  border: 0;
}

.tfo-notebook-buttons td {
  padding-left: 0;
  padding-right: 20px;
}

.tfo-notebook-buttons {
  --tfo-notebook-buttons-box-shadow: 0 1px 2px 0 rgba(60, 64, 67, .3), 0 1px 3px 1px rgba(60, 64, 67, .15);
}

.tfo-notebook-buttons a,
.tfo-notebook-buttons :link,
.tfo-notebook-buttons :visited {
  border-radius: 8px;
  box-shadow: var(--tfo-notebook-buttons-box-shadow);
  color: #202124;
  padding: 12px 24px;
  transition: box-shadow 0.2s;
}

.tfo-notebook-buttons a:hover,
.tfo-notebook-buttons a:focus {
  box-shadow: var(--tfo-notebook-buttons-box-shadow);
}

.tfo-notebook-buttons td > a {
  -webkit-box-align: center;
  -ms-flex-align: center;
  align-items: center;
  display: -webkit-box;
  display: -ms-flexbox;
  display: flex;
}

.tfo-notebook-buttons td > a > img {
  margin-right: 8px;
}
</style>

<table class="tfo-notebook-buttons tfo-api nocontent" align="left">
  <tbody>
    <tr>
      <td>
        <a target="_blank" href="https://colab.research.google.com/github/google/meridian/blob/main/demo/Meridian_Getting_Started.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
      </td>
      <td>
        <a target="_blank" href="https://github.com/google/meridian/blob/main/demo/Meridian_Getting_Started.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
      </td>
    </tr>
  </tbody>
</table>

# **Introduction to Meridian Demo**

Welcome to the Meridian end-to-end demo. This simplified demo showcases the fundamental functionalities and basic usage of the library, including working examples of the major modeling steps:


<ol start="0">
  <li><a href="#install">Install and Environment Configuration</a></li>
  <li><a href="#load-data">Load the data</a></li>
  <li><a href="#configure-model">Configure the model</a></li>
  <li><a href="#run-eda">Run exploratory data analysis & two-pager output</a></li>
  <li><a href="#fit-model">Fit the model</a></li>
  <li><a href="#quality-checks">Run post-modeling health checks</a></li>
  <li><a href="#model-diagnostics">Run model diagnostics</a></li>
  <li><a href="#generate-summary">Generate model results & two-page output</a></li>
  <li><a href="#generate-optimize">Run budget optimization & two-page output</a></li>
  <li><a href="#save-model">Save the model object</a></li>
  <li><a href="#scenario-planning">Interactive Scenario Planning</a></li>
</ol>


Note that this notebook assumes that you have completed basic preprocessing steps before reaching this point in the demo.

This notebook utilizes sample data. As a result, the numbers and results obtained might not accurately reflect what you encounter when working with a real dataset.

You can also run this demo using the [JAX backend](https://developers.google.com/meridian/notebook/meridian-getting-started-jax).

<a name="install"></a>
## Step 0: Install and Environment Configuration

1\. Make sure you are using one of the available GPU Colab runtimes which is **required** to run Meridian. You can change your notebook's runtime in `Runtime > Change runtime type` in the menu. All users can use the T4 GPU runtime which is sufficient to run the demo colab, free of charge. Users who have purchased one of Colab's paid plans have access to premium GPUs (such as V100, A100 or L4 Nvidia GPU).

2\. Install the latest version of Meridian, and verify that GPU is available.

In [4]:
# Install meridian: from PyPI @ latest release
!pip install --upgrade google-meridian[colab,and-cuda,schema]

# Install meridian: from PyPI @ specific version
# !pip install google-meridian[colab,and-cuda,schema]==1.3.1

# Install meridian: from GitHub @HEAD
# !pip install --upgrade "google-meridian[colab,and-cuda,schema] @ git+https://github.com/google/meridian.git@main"

In [5]:
import IPython
from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.analysis.review import reviewer
from meridian.data import data_frame_input_data_builder
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
from meridian.model.eda import meridian_eda
from meridian.schema.serde import meridian_serde
import numpy as np
import pandas as pd
# check if GPU is available
from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp

ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print(
    'Num GPUs Available: ',
    len(tf.config.experimental.list_physical_devices('GPU')),
)
print(
    'Num CPUs Available: ',
    len(tf.config.experimental.list_physical_devices('CPU')),
)

Your runtime has 13.6 gigabytes of available RAM

Num GPUs Available:  0
Num CPUs Available:  1


3\. Mount a storage. Use `meridian_root` to refer to the mounted root. The mounted root will be used to <a href="#save-model">save trained model</a>, <a href="#generate-summary">stage two-pager output</a> and <a href="#scenario-planning">generate scenario planning dashboard</a>.

For Colab Free/Pro user, we will use the `MyDrive` folder in Google Drive as the external storage. For Colab Enterprise user, we will use <a href="https://docs.cloud.google.com/storage/docs/cloud-storage-fuse/overview">Cloud FUSE</a> to mount a GCS bucket.

In [6]:
# @markdown If you are using Colab Free, Colab Pro, run this cell to mount your Google Drive.
from google.colab import drive
drive_mount = '/content/drive'
drive.mount(drive_mount, force_remount=True)
subfolder = '' # @param {"type":"string","placeholder": "Optional, specifying a subfolder is recommended for organizing distinct execution runs."}
# Change this "MyDrive" to other share folders name if you would like to use a different drive.
meridian_root = f'{drive_mount}/MyDrive/{subfolder}'
is_enterprise_user=False

Mounted at /content/drive


In [ ]:
# @markdown If you are using Colab Enterprise, uncomment and run this cell to mount a GCS bucket.
# import os

# project_id = ""# @param {"type":"string","placeholder": "Cloud project id"}
# bucket_name = "" # @param {"type":"string","placeholder": "GCS bucket that contains your model"}
# subfolder = "" # @param {"type":"string","placeholder": "Optional, specifying a subfolder is recommended for organizing distinct execution runs."}
# os.environ['GOOGLE_CLOUD_PROJECT'] = project_id
# !gcloud config set project {project_id}
# !gcloud auth login
# !mkdir /content/{bucket_name}

# # Uncomment below if you don't have gcsfuse installed
# #!echo "deb https://packages.cloud.google.com/apt gcsfuse-`lsb_release -c -s` main" | sudo tee /etc/apt/sources.list.d/gcsfuse.list
# #!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -
# #!apt-get update
# #!apt-get install gcsfuse

# !gcsfuse --implicit-dirs {bucket_name} /content/{bucket_name}
# meridian_root = f'/content/{bucket_name}/{subfolder}'
# is_enterprise_user=True
#!fusermount -u /content/{bucket_name}

<a name="load-data"></a>
## Step 1: Load the data

Load the [simulated dataset in CSV format](https://github.com/google/meridian/blob/main/meridian/data/simulated_data/csv/geo_all_channels.csv) as follows.

1\. Read the data into a Pandas DataFrame.

In [7]:
df = pd.read_csv(
    # Optionally, use `f"${meridian_root}/<path_to_csv>"` to load data from the mounted storage.
    "https://raw.githubusercontent.com/google/meridian/refs/heads/main/meridian/data/simulated_data/csv/geo_all_channels.csv",
    index_col=0
)

In [8]:
# Aggregate to national level
channels = ["Channel0", "Channel1", "Channel2", "Channel3", "Channel4"]
raw_df = df.copy()
raw_df["revenue"] = raw_df["conversions"] * raw_df["revenue_per_conversion"]

agg_dict = {col: "sum" for col in
    [f"{ch}_impression" for ch in channels] +
    [f"{ch}_spend" for ch in channels] +
    ["Organic_channel0_impression", "conversions", "revenue"]}
agg_dict["Promo"] = "mean"

national_df = raw_df.groupby("time").agg(agg_dict).reset_index()
national_df["revenue_per_conversion"] = national_df["revenue"] / national_df["conversions"]
national_df.drop(columns=["revenue"], inplace=True)

print(f"National dataset: {national_df.shape[0]} weeks")
national_df.head()

National dataset: 156 weeks


,time,Channel0_impression,Channel1_impression,Channel2_impression,Channel3_impression,Channel4_impression,Channel0_spend,Channel1_spend,Channel2_spend,Channel3_spend,Channel4_spend,Organic_channel0_impression,conversions,Promo,revenue_per_conversion
0,2021-01-25,44340630,10537596,6501889,66738506,53026011,325137.56830,101595.404270,48315.04518,520082.5242,413173.984050,24028870,386300926.3,0.234740,0.020015
1,2021-02-01,47391940,30857600,7966320,79000018,27592214,347511.96809,297505.263890,59197.12148,615634.5403,214996.093200,29188267,355303931.3,0.145423,0.019996
2,2021-02-08,28960185,23144749,4637739,68442555,20057461,212357.01945,223143.885196,34462.68770,533361.9165,156285.964472,12051369,438308889.8,0.537338,0.019992
3,2021-02-15,38126016,10667077,3029637,70601091,37246623,279567.52290,102843.758830,22513.00335,550183.0380,290222.395240,9595212,418612413.7,0.367233,0.019979
4,2021-02-22,46890794,17925961,10486840,81842660,56259530,343837.20920,172828.342330,77926.91530,637786.8024,438369.280950,9288846,355785298.0,0.429602,0.019983


### Example: Dataset preview and schema overview

A preview of the aggregated dataset is shown below, as well as a basic data dictionary. For more details on the data requirements for Meridian, check out [Collect and organize your data](https://developers.google.com/meridian/docs/pre-modeling/collect-data).

In [9]:
# An illustrative example of the dataset used by Meridian is shown below.
# Note that for each given geo, a series of aggregated values is given
# for each specific time period (in this example, week)
preview_national_df = national_df[national_df['time'].isin(['2021-01-25', '2021-02-01', '2021-02-08'])].head(6)  # [rk] df -> national_df
IPython.display.HTML(
    f'<div style="overflow-x: auto;">'
    f'{preview_national_df.to_html()}'
    f'</div>'
)

,time,Channel0_impression,Channel1_impression,Channel2_impression,Channel3_impression,Channel4_impression,Channel0_spend,Channel1_spend,Channel2_spend,Channel3_spend,Channel4_spend,Organic_channel0_impression,conversions,Promo,revenue_per_conversion
0,2021-01-25,44340630,10537596,6501889,66738506,53026011,325137.56830,101595.404270,48315.04518,520082.5242,413173.984050,24028870,386300926.3,0.234740,0.020015
1,2021-02-01,47391940,30857600,7966320,79000018,27592214,347511.96809,297505.263890,59197.12148,615634.5403,214996.093200,29188267,355303931.3,0.145423,0.019996
2,2021-02-08,28960185,23144749,4637739,68442555,20057461,212357.01945,223143.885196,34462.68770,533361.9165,156285.964472,12051369,438308889.8,0.537338,0.019992


**Dataset legend:**

<table>
  <tr>
    <th width="30%">Column</th>
    <th width="20%">Variable type</th>
    <th width="50%">Description</th>
  </tr>
  <tr>
    <td>time</td>
    <td>Time</td>
    <td>The specific time period for the data row (for example, weekly).</td>
  </tr>
  <tr>
    <td>geo</td>
    <td>Geography</td>
    <td>The geographic region for the observation (for example, Designated Market Area, state).</td>
  </tr>
  <tr>
    <td>conversions</td>
    <td>KPI</td>
    <td>The model's response variable. For example total sales or sign-ups.</td>
  </tr>
  <tr>
    <td>revenue_per_conversion</td>
    <td>Revenue per KPI</td>
    <td>The average revenue generated per KPI unit.</td>
  </tr>
  <tr>
    <td>Channel0_spend, Channel1_spend, Channel2_spend, Channel3_spend, Channel4_spend</td>
    <td>Media spend</td>
    <td>The total media spend for your specific paid channels during that week and geo.</td>
  </tr>
  <tr>
    <td>Channel0_impression, Channel1_impression, Channel2_impression, Channel3_impression, Channel4_impression</td>
    <td>Media data</td>
    <td>The media execution metric (for example, impressions, clicks) for those paid channels.</td>
  </tr>
  <tr>
    <td>Organic_channel0_impression</td>
    <td>Organic media</td>
    <td>An organic media metric that drives outcomes but has no direct media cost.</td>
  </tr>
  <tr>
    <td>competitor_sales_control, sentiment_score_control, Promo</td>
    <td>Control variables</td>
    <td>Control variables (confounders) that affect both your outcome and your media decisions.</td>
  </tr>
  <tr>
    <td>population</td>
    <td>Population</td>
    <td>The population of the geographic region.</td>
  </tr>
</table>


2\. Create a DataFrameInputDataBuilder instance.

In [10]:
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type='non_revenue',
    default_kpi_column='conversions',
    default_revenue_per_kpi_column='revenue_per_conversion',
)

3\. Offer the components to the builder. Note that the components may be offered all at once or piecewise.

In [12]:
# builder = (
#     builder.with_kpi(df)
#     .with_revenue_per_kpi(df)
#     .with_population(df)
#     .with_controls(
#         df, control_cols=["sentiment_score_control", "competitor_sales_control"]
#     )
# )

# COMMENTED OUT .with_population(), and the two control variables (sentiment_score_control, competitor_sales_control).

builder = (
    builder
    .with_kpi(national_df)
    .with_revenue_per_kpi(national_df)
)


channels = ["Channel0", "Channel1", "Channel2", "Channel3", "Channel4"]
builder = builder.with_media(
    national_df,
    media_cols=[f"{ch}_impression" for ch in channels],
    media_spend_cols=[f"{ch}_spend" for ch in channels],
    media_channels=channels,
)

4. If your data includes organic media or non-media treatments, you can add them using `with_organic_media` and `with_non_media_treatments` methods. For the definition of each variable, see
[Collect and organize your data](https://developers.google.com/meridian/docs/pre-modeling/collect-data)

In [13]:
builder = builder.with_non_media_treatments(
    national_df, non_media_treatment_cols=['Promo']
).with_organic_media(
    national_df,
    organic_media_cols=['Organic_channel0_impression'],
    organic_media_channels=['Organic_channel0'],
)

5. Finally, build the InputData.

In [14]:
data = builder.build()

Note that the simulated data here does not contain reach and frequency. We recommend including reach and frequency data whenever they are available. For information about the advantages of utilizing reach and frequency, see [Bayesian Hierarchical Media Mix Model Incorporating Reach and Frequency Data](https://research.google/pubs/bayesian-hierarchical-media-mix-model-incorporating-reach-and-frequency-data/#:~:text=By%20incorporating%20R%26F%20into%20MMM,based%20on%20optimal%20frequency%20recommendations.). For code snippet for loading reach and frequency data, see [Load geo-level data with reach and frequency](https://developers.google.com/meridian/docs/user-guide/load-geo-data-with-rf)

The documentation provides guidance for instances where reach and frequency data is accessible for specific channels. Additionally, for information about how to load other data types and formats, including data with reach and frequency, see [Supported data types and formats](https://developers.google.com/meridian/docs/user-guide/supported-data-types-formats).

## [rk] Add holdout


In [15]:
n_times = len(data.time)
holdout_id = np.full(n_times, False)
holdout_id[-13:] = True  # last quarter
print(f"Train: {(~holdout_id).sum()} weeks | Test: {holdout_id.sum()} weeks")

Train: 143 weeks | Test: 13 weeks


<a name="configure-model"></a>
## Step 2: Configure the model

Meridian uses Bayesian framework and Markov Chain Monte Carlo (MCMC) algorithms to sample from the posterior distribution.

1\. Inititalize the `Meridian` class by passing the loaded data and the customized model specification. One advantage of Meridian lies in its capacity to calibrate the model directly through ROI priors, as described in [Media Mix Model Calibration With Bayesian Priors](https://research.google/pubs/media-mix-model-calibration-with-bayesian-priors/). In this particular example, the ROI priors for all media channels are identical, with each being represented as Lognormal(0.2, 0.9).

In [16]:
roi_mu = 0.2  # Mu for ROI prior for each media channel.
roi_sigma = 0.9  # Sigma for ROI prior for each media channel.
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(roi_mu, roi_sigma, name=constants.ROI_M)
)
model_spec = spec.ModelSpec(prior=prior, enable_aks=True, holdout_id=holdout_id) # [rk] Added holdout_id

mmm = model.Meridian(input_data=data, model_spec=model_spec)

/usr/local/lib/python3.12/dist-packages/meridian/model/model.py:103: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(


2\. Use the `sample_prior()` method to obtain samples from the prior distributions of model parameters.

In [17]:
mmm.sample_prior(500)

/usr/local/lib/python3.12/dist-packages/meridian/model/knots.py:614: RuntimeWarning: overflow encountered in cast
  backend.np_float_dtype(math.comb(ncol, design_mat.shape[1]))
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1391: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. tau_g_excl_baseline has been automatically set to Deterministic(0).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1391: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_m has been automatically set to Deterministic(0).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1391: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_rf has been automatically set to Deterministic(0).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/

<a name="run-eda"></a>
## Step 3: Run exploratory data analysis & two-pager output

Meridian EDA helps identify potential issues with your dataset before fitting the model.
To obtain the EDA HTML report, initialize `MeridianEDA` with the `Meridian` model object and call `generate_and_save_report`.

Note: `MeridianEDA` requires prior samples to run. It will automatically run prior sampling if the model hasn't run it yet.

In [18]:
eda = meridian_eda.MeridianEDA(mmm)
filename = 'eda_report.html'
eda.generate_and_save_report(filename=filename, filepath=meridian_root)
IPython.display.HTML(filename=f'{meridian_root}/{filename}')

For more information about Meridian's EDA checks, visualizations, and user customizations, see the [Meridian's EDA package](https://developers.google.com/meridian/docs/pre-modeling/perform-eda#meridians_eda_package).

<a name="fit-model"></a>
## Step 4: Fit the model

Use the `sample_posterior()` method to obtain samples from the posterior distributions of model parameters. If you are using the T4 GPU runtime this step may take about 10 minutes for the provided data set.

In [19]:
%%time
mmm.sample_posterior(
    n_chains=10, n_adapt=2000, n_burnin=500, n_keep=1000, seed=0
)

CPU times: user 10min 53s, sys: 26.9 s, total: 11min 20s
Wall time: 9min 35s


/usr/local/lib/python3.12/dist-packages/arviz/data/inference_data.py:157: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/arviz/data/inference_data.py:1647: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(


For more information about configuring the parameters and using a customized model specification, such as setting different ROI priors for each media channel, see [Configure the model](https://developers.google.com/meridian/docs/user-guide/configure-model).

<a name="quality-checks"></a>
## Step 5: Run post-modeling health checks

These post-modeling health checks are designed to diagnose common issues related to model convergence, specification, and plausibility. Run the following command to generate the results for all necessary diagnostics:

In [20]:
health_summary = reviewer.ModelReviewer(mmm).run()

filename = 'health_card.html'
health_summary.output_model_health_card(filename=filename, filepath=meridian_root)
IPython.display.HTML(filename=f'{meridian_root}{filename}')

/tmp/ipykernel_2019/879150696.py:1: DeprecationWarning: The `meridian` argument is deprecated. Please use `model_context` and `inference_data` instead.
  health_summary = reviewer.ModelReviewer(mmm).run()
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:103: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(


Metric check,Status,Recommended action
Convergence,Pass,"The model has likely converged, as all parameters have R-hat values < 1.2."
Baseline,Pass,The posterior probability that the baseline is negative is 0.00. We recommend visually inspecting the baseline time series in the Model Fit charts to confirm this.
Bayesian p-value,Pass,The Bayesian posterior predictive p-value is 0.74. The observed total outcome is consistent with the model's posterior predictive distribution.
Goodness of fit,Review,"R-squared = 0.8472 (All), 0.9315 (Train), -0.9715 (Test); MAPE = 0.0178 (All), 0.0135 (Train), 0.0646 (Test); wMAPE = 0.0180 (All), 0.0136 (Train), 0.0635 (Test). A negative R-squared signals a potential conflict between your priors and the data, and it warrants investigation. If this conflict is intentional (due to an informative prior), no further action is needed. If it's unintentional, we recommend relaxing your priors to be less restrictive."
Prior-posterior shift,Pass 5/5 channels passed,The model has successfully learned from the data. This is a positive sign that your data was informative.


<a name="model-diagnostics"></a>
## Step 6: Run model diagnostics

To further assess convergence and model fit, you can use the methods from `visualizer` module.

1\. Assess convergence. Run the following code to generate r-hat statistics. R-hat close to 1.0 indicate convergence. R-hat < 1.2 indicates approximate convergence and is a reasonable threshold for many problems.

In [21]:
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()
model_diagnostics.predictive_accuracy_table() # [rk] Added for holdout metrics

/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:103: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(


,metric,geo_granularity,evaluation_set,value
0,R_Squared,national,Train,0.931548
1,R_Squared,national,Test,-0.971471
2,R_Squared,national,All Data,0.847177
3,MAPE,national,Train,0.013503
4,MAPE,national,Test,0.064626
5,MAPE,national,All Data,0.017763
6,wMAPE,national,Train,0.013627
7,wMAPE,national,Test,0.063532
8,wMAPE,national,All Data,0.017972


2\. Assess the model's fit by comparing the expected sales against the actual sales.

In [22]:
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()

/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:103: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(


alt.LayerChart(...)

For more information and additional model diagnostics checks, see [Modeling diagnostics](https://developers.google.com/meridian/docs/user-guide/model-diagnostics).

<a name="generate-summary"></a>
## Step 7: Generate model results & two-page output

To export the two-page HTML summary output, initialize the `Summarizer` class with the model object. Then pass in the filename, filepath, start date, and end date to `output_model_results_summary` to run the summary for that time duration and save it to the specified file.

In [23]:
mmm_summarizer = summarizer.Summarizer(mmm)

In [24]:
filepath = meridian_root
start_date = '2021-01-25'
end_date = '2024-01-15'
mmm_summarizer.output_model_results_summary(
    'summary_output.html', filepath, start_date, end_date
)

/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:103: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:103: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:2401: UserWarning: Effectiveness is not reported because it does not have a clear interpretation by time period.
  warnings.warn(


Here is a preview of the two-page output based on the simulated data:

In [25]:
IPython.display.HTML(filename=f'{meridian_root}/summary_output.html')

Dataset,R-squared,MAPE,wMAPE
Training Data,0.93,1%,1%
Testing Data,-0.97,6%,6%
All Data,0.85,2%,2%


For a customized two-page report, model results summary table, and individual visualizations, see [Model results report](https://developers.google.com/meridian/docs/user-guide/generate-model-results-report) and [plot media visualizations](https://developers.google.com/meridian/docs/user-guide/plot-media-visualizations).





<a name="generate-optimize"></a>
## Step 8: Run budget optimization & generate an optimization report

You can choose what scenario to run for the budget allocation. In default scenario, you find the optimal allocation across channels for a given budget to maximize the return on investment (ROI).

Alternatively, if you would like to have a sharable interactive dashboard, check out [Meridian Scenario Planner](https://developers.google.com/meridian/docs/scenario-planning/meridian-scenario-planner).

1\. Instantiate the `BudgetOptimizer` class and run the `optimize()` method without any customization, to run the default library's Fixed Budget Scenario to maximize ROI.

In [28]:
%%time
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()

CPU times: user 2min 24s, sys: 1.32 s, total: 2min 25s
Wall time: 1min 47s


2\. Export the 2-page HTML optimization report, which contains optimized spend allocations and ROI.

In [29]:
filepath = meridian_root
optimization_results.output_optimization_summary(
    'optimization_output.html', filepath
)

In [30]:
IPython.display.HTML(filename=f'{meridian_root}/optimization_output.html')

Channel,Non-optimized spend,Optimized spend
Channel3,40%,28%
Channel0,18%,23%
Channel4,22%,23%
Channel1,14%,19%
Channel2,6%,7%


For information about customized optimization scenarios, such as flexible budget scenarios, see [Budget optimization scenarios](https://developers.google.com/meridian/docs/user-guide/budget-optimization-scenarios). For more information about optimization results summary and individual visualizations, see [optimization results output](https://developers.google.com/meridian/docs/user-guide/generate-optimization-results-output) and [optimization visualizations](https://developers.google.com/meridian/docs/user-guide/plot-optimization-visualizations).


Optimization can also be performed on a hypothetical data representing a future scenario. The new data takes the same structure as the input data and encodes an anticipated flighting pattern, cost per media unit, and revenue per kpi.

3\. Load the [simulated dataset in CSV format](https://github.com/google/meridian/blob/main/meridian/data/simulated_data/csv/hypothetical_geo_all_channels.csv) into Pandas DataFrame.

In [31]:
national_df = pd.read_csv(
    "https://raw.githubusercontent.com/google/meridian/refs/heads/main/meridian/data/simulated_data/csv/hypothetical_geo_all_channels.csv"
)



4\. New data is read from a csv file and converted into a set of multi-dimensional arrays. The arrays are used to construct a `DataTensors` instance, which is passed to `optimize()` as the `new_data` argument.

Constructing a `DataTensors` instance requires that all arrays have "time" and "geo" dimensions. Alternatively, the `BudgetOptimizer.create_optimization_tensors` method can be used to construct a `DataTensors` instance. This helper method can simplify the process, particularly when you do not need "time" and "geo" dimensions for all inputs. For example, it can be convenient if you want to assume a constant "revenue per kpi" or "cost per media unit" for all geos and time periods.

In [32]:
n_geos = mmm.n_geos
n_media_channels = mmm.n_media_channels
n_non_media_channels = mmm.n_non_media_channels
n_organic_media_channels = mmm.n_organic_media_channels

# The number of time periods and time range do not need to match the input data.
national_df[constants.TIME] = pd.to_datetime(national_df[constants.TIME], errors='coerce')
unique_times = sorted(national_df[constants.TIME].unique())
n_times = len(unique_times)

geos = mmm.input_data.geo.values
media_channels = mmm.input_data.media_channel.values
media_cols = [f"{channel}_impression" for channel in media_channels]
media_spend_cols = [f"{channel}_spend" for channel in media_channels]
non_media_treatment_cols = ['Promo']
organic_media_cols = ['Organic_channel0_impression']
organic_media_channels = ['Organic_channel0']
revenue_per_kpi_col='revenue_per_conversion'
times_str = [time.strftime(constants.DATE_FORMAT) for time in unique_times]

media_np = np.zeros((n_geos, n_times, n_media_channels))
media_spend_np = np.zeros((n_geos, n_times, n_media_channels))
non_media_treatment_np = np.zeros((n_geos, n_times, n_non_media_channels))
organic_media_np = np.zeros((n_geos, n_times, n_organic_media_channels))
revenue_per_kpi_np = np.zeros((n_geos, n_times))

national_df_grouped = national_df.set_index([constants.GEO, constants.TIME])
# for geo_idx, geo in enumerate(geos):
#   for time_idx, time in enumerate(unique_times):
#     row = national_df_grouped.loc[(geo, time)]
#     media_np[geo_idx, time_idx, :] = row[media_cols].values
#     media_spend_np[geo_idx, time_idx, :] = row[media_spend_cols].values
#     non_media_treatment_np[geo_idx, time_idx, :] = row[non_media_treatment_cols].values
#     organic_media_np[geo_idx, time_idx, :] = row[organic_media_cols].values
#     revenue_per_kpi_np[geo_idx, time_idx] = row[revenue_per_kpi_col].item()

data_tensors = analyzer.DataTensors(
    media=tf.convert_to_tensor(media_np, dtype=tf.float32),
    media_spend=tf.convert_to_tensor(media_spend_np, dtype=tf.float32),
    non_media_treatments=tf.convert_to_tensor(non_media_treatment_np, dtype=tf.float32),
    organic_media=tf.convert_to_tensor(organic_media_np, dtype=tf.float32),
    revenue_per_kpi=tf.convert_to_tensor(revenue_per_kpi_np, dtype=tf.float32),
    time=tf.convert_to_tensor(times_str, dtype=tf.string),
)
# Default values for `budget` and `pct_of_spend` are derived from the `new_data`,
# but these values can be overridden without modifying the `new_data` itself.
hypothetical_optimization_results = budget_optimizer.optimize(
    new_data=data_tensors,
    budget=50_000_000,
    pct_of_spend=[.2, .1, .2, .2, .3]
)

/usr/local/lib/python3.12/dist-packages/meridian/analysis/tensors.py:715: UserWarning: A `organic_media` value was passed in the `new_data` argument. This is not supported and will be ignored.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/analysis/tensors.py:715: UserWarning: A `non_media_treatments` value was passed in the `new_data` argument. This is not supported and will be ignored.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:4779: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


5\. Export the 2-page HTML optimization report.

In [33]:
filepath = meridian_root
hypothetical_optimization_results.output_optimization_summary(
    'hypothetical_optimization_output.html', filepath
)

/usr/local/lib/python3.12/dist-packages/meridian/analysis/tensors.py:715: UserWarning: A `organic_media` value was passed in the `new_data` argument. This is not supported and will be ignored.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/analysis/tensors.py:715: UserWarning: A `non_media_treatments` value was passed in the `new_data` argument. This is not supported and will be ignored.
  warnings.warn(


In [34]:
IPython.display.HTML(filename=f'{meridian_root}/hypothetical_optimization_output.html')

Channel,Non-optimized spend,Optimized spend
Channel4,30%,27%
Channel0,20%,20%
Channel2,20%,20%
Channel3,20%,20%
Channel1,10%,13%


<a name="save-model"></a>
## Step 9: Save the model object

We recommend that you save the model object for future use. This helps you to  avoid repetitive model runs and saves time and computational resources. After the model object is saved, you can load it at a later stage to continue the analysis or visualizations without having to re-run the model.


Run the following codes to save the model object:

In [35]:
file_path = f'{meridian_root}/saved_mmm.binpb'
meridian_serde.save_meridian(mmm, file_path)
print(f'model is saved at {file_path}')

model is saved at /content/drive/MyDrive//saved_mmm.binpb


Run the following codes to load the saved model:

In [36]:
mmm = meridian_serde.load_meridian(file_path)

/usr/local/lib/python3.12/dist-packages/arviz/data/inference_data.py:1538: UserWarning: The group trace is not defined in the InferenceData scheme
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/model.py:103: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/knots.py:614: RuntimeWarning: overflow encountered in cast
  backend.np_float_dtype(math.comb(ncol, design_mat.shape[1]))


In [37]:
# @markdown If a GCS bucket is mounted, run this cell to unmount it.
!fusermount -u /content/{bucket_name}

fusermount: failed to unmount /content/{bucket_name}: No such file or directory


<a name="scenario-planning"></a>
## Step 10: Interactive Scenario Planning
[Meridian Scenario Planner](https://developers.google.com/meridian/docs/scenario-planning/meridian-scenario-planner) is a tool that allow advertisers to create a sharable and interactive dashboard from Colab; you can reuse the saved model from this Colab for dashboard generation.